# Approach 3.2 — Controlled Vocabulary Decoder

## What this notebook adds over 03_2

Notebook `03_2` trains an LSTM decoder over the full open vocabulary (~20 000 tokens).  
This notebook trains the same decoder but constrains output to the **top-1000 most frequent tokens** from the training corpus.

The attacker's assumption : the victim's sentences are drawn from the same domain distribution (professional bios, personal facts), so a small closed vocabulary is enough to reconstruct the semantically important words.

The training data and precomputed embeddings are the same as in notebooks 01–03_1 — no re-encoding needed.

In [ ]:
import copy
import faiss
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

TRAIN_DATA_FILE   = 'data/sentences_train_text_db.parquet'
TRAIN_INDEX_FILE  = 'data/sentences_train_vector_db.index'
TARGET_DATA_FILE  = 'data/sentences_target_text_db.parquet'
TARGET_INDEX_FILE = 'data/sentences_target_vector_db.index'
TOP_VOCAB_SIZE    = 1000
HIDDEN_DIM        = 256
TOKEN_EMB_DIM     = 64
MAX_LEN           = 30
BATCH_SIZE        = 256
EPOCHS            = 30
PATIENCE          = 5
LR                = 1e-3
VAL_SIZE          = 0.2
RANDOM_SEED       = 42
SAMPLE_EVERY      = 5

PAD_TOKEN = '<PAD>'
BOS_TOKEN = '<BOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'

torch.manual_seed(RANDOM_SEED)
# LSTM on MPS triggers a kernel OOM crash — use CPU which is stable for this model size
DEVICE = torch.device('cpu')
print(f'device : {DEVICE}')

In [ ]:
df          = pd.read_parquet(TRAIN_DATA_FILE)
faiss_index = faiss.read_index(TRAIN_INDEX_FILE)
all_emb     = np.zeros((faiss_index.ntotal, faiss_index.d), dtype=np.float32)
faiss_index.reconstruct_n(0, faiss_index.ntotal, all_emb)

SENTENCE_EMB_DIM = faiss_index.d
embeddings       = all_emb[df['id'].values]
sentences        = df['text'].tolist()
del all_emb

print(f'sentences : {len(sentences):,}')
print(f'emb shape : {embeddings.shape}')

Embeddings are loaded from the precomputed FAISS index — no re-encoding needed.

In [ ]:
word_freq   = Counter(tok for sent in sentences for tok in sent.split())
top_words   = [w for w, _ in word_freq.most_common(TOP_VOCAB_SIZE)]
vocab       = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN] + sorted(top_words)
token_to_id = {tok: i for i, tok in enumerate(vocab)}
id_to_token = {i: tok for tok, i in token_to_id.items()}

PAD_ID     = token_to_id[PAD_TOKEN]
BOS_ID     = token_to_id[BOS_TOKEN]
EOS_ID     = token_to_id[EOS_TOKEN]
UNK_ID     = token_to_id[UNK_TOKEN]
VOCAB_SIZE = len(vocab)

total_tokens   = sum(len(s.split()) for s in sentences)
covered_tokens = sum(1 for s in sentences for t in s.split() if t in token_to_id)
print(f'vocab size : {VOCAB_SIZE}')
print(f'coverage   : {covered_tokens / total_tokens:.1%} of all tokens in training data')


def encode_sentence(sentence: str) -> tuple[list[int], list[int]]:
    tids       = [token_to_id.get(t, UNK_ID) for t in sentence.split()]
    dec_input  = ([BOS_ID] + tids)[:MAX_LEN]
    dec_target = (tids + [EOS_ID])[:MAX_LEN]
    dec_input  += [PAD_ID] * (MAX_LEN - len(dec_input))
    dec_target += [PAD_ID] * (MAX_LEN - len(dec_target))
    return dec_input, dec_target


def decode_ids(ids: list[int]) -> str:
    tokens = []
    for i in ids:
        if i == EOS_ID:
            break
        if i not in (PAD_ID, BOS_ID):
            tokens.append(id_to_token[i])
    return ' '.join(tokens)

In [ ]:
class SentenceEmbeddingDataset(Dataset):
    def __init__(self, embeddings: np.ndarray, sentences: list[str]):
        self.embeddings  = torch.from_numpy(np.ascontiguousarray(embeddings))
        encoded          = [encode_sentence(s) for s in sentences]
        self.dec_inputs  = torch.tensor([e[0] for e in encoded], dtype=torch.long)
        self.dec_targets = torch.tensor([e[1] for e in encoded], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.embeddings)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.embeddings[idx], self.dec_inputs[idx], self.dec_targets[idx]


emb_train, emb_val, sent_train, sent_val = train_test_split(
    embeddings, sentences, test_size=VAL_SIZE, random_state=RANDOM_SEED
)

train_loader = DataLoader(SentenceEmbeddingDataset(emb_train, sent_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SentenceEmbeddingDataset(emb_val,   sent_val),   batch_size=BATCH_SIZE, shuffle=False)

print(f'train: {len(sent_train):,} | val: {len(sent_val):,}')

In [ ]:
class LSTMDecoder(nn.Module):
    def __init__(self, sentence_emb_dim: int, hidden_dim: int, token_emb_dim: int, vocab_size: int):
        super().__init__()
        self.proj      = nn.Linear(sentence_emb_dim, hidden_dim)
        self.token_emb = nn.Embedding(vocab_size, token_emb_dim, padding_idx=PAD_ID)
        self.lstm      = nn.LSTM(token_emb_dim, hidden_dim, batch_first=True)
        self.out       = nn.Linear(hidden_dim, vocab_size)

    def forward(self, sentence_emb: torch.Tensor, dec_input_ids: torch.Tensor) -> torch.Tensor:
        h0     = self.proj(sentence_emb).unsqueeze(0)
        c0     = torch.zeros_like(h0)
        tok_e  = self.token_emb(dec_input_ids)
        out, _ = self.lstm(tok_e, (h0, c0))
        return self.out(out)


decoder   = LSTMDecoder(SENTENCE_EMB_DIM, HIDDEN_DIM, TOKEN_EMB_DIM, VOCAB_SIZE).to(DEVICE)
optimizer = torch.optim.Adam(decoder.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

print(f'parameters : {sum(p.numel() for p in decoder.parameters()):,}')

In [ ]:
def greedy_decode(emb: torch.Tensor) -> str:
    decoder.eval()
    with torch.no_grad():
        hx     = decoder.proj(emb).unsqueeze(0)
        cx     = torch.zeros_like(hx)
        cur_id = torch.tensor([[BOS_ID]], dtype=torch.long, device=DEVICE)
        result = []
        for _ in range(MAX_LEN):
            tok_e         = decoder.token_emb(cur_id)
            out, (hx, cx) = decoder.lstm(tok_e, (hx, cx))
            next_id       = decoder.out(out.squeeze(1)).argmax(dim=-1).item()
            if next_id == EOS_ID:
                break
            result.append(next_id)
            cur_id = torch.tensor([[next_id]], dtype=torch.long, device=DEVICE)
    return ' '.join(id_to_token[i] for i in result if i not in (PAD_ID, BOS_ID))


sample_emb = torch.tensor(emb_val[0], dtype=torch.float32).unsqueeze(0).to(DEVICE)
sample_ref = sent_val[0]

best_val_loss  = float('inf')
patience_count = 0
best_weights   = None

for epoch in range(1, EPOCHS + 1):
    decoder.train()
    total_train = 0.0
    for emb_batch, dec_in, dec_tgt in train_loader:
        emb_batch = emb_batch.to(DEVICE)
        dec_in    = dec_in.to(DEVICE)
        dec_tgt   = dec_tgt.to(DEVICE)
        logits    = decoder(emb_batch, dec_in)
        loss      = criterion(logits.view(-1, VOCAB_SIZE), dec_tgt.view(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_train += loss.item()

    avg_train = total_train / len(train_loader)

    decoder.eval()
    total_val = 0.0
    with torch.no_grad():
        for emb_batch, dec_in, dec_tgt in val_loader:
            emb_batch = emb_batch.to(DEVICE)
            dec_in    = dec_in.to(DEVICE)
            dec_tgt   = dec_tgt.to(DEVICE)
            logits    = decoder(emb_batch, dec_in)
            total_val += criterion(logits.view(-1, VOCAB_SIZE), dec_tgt.view(-1)).item()
    avg_val = total_val / len(val_loader)

    if avg_val < best_val_loss:
        best_val_loss  = avg_val
        patience_count = 0
        best_weights   = copy.deepcopy(decoder.state_dict())
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'early stop at epoch {epoch}  —  best val {best_val_loss:.4f}')
            break

    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        preview = greedy_decode(sample_emb)
        print(f'epoch {epoch:3d} | train {avg_train:.4f} | val {avg_val:.4f} | "{preview}"')
        print(f'ref : "{sample_ref}"')
    else:
        print(f'epoch {epoch:3d} | train {avg_train:.4f} | val {avg_val:.4f}')

decoder.load_state_dict(best_weights)
print(f'restored best weights (val {best_val_loss:.4f})')

In [ ]:
target_meta_df = pd.read_parquet(TARGET_DATA_FILE)
target_fi      = faiss.read_index(TARGET_INDEX_FILE)
target_all_emb = np.zeros((target_fi.ntotal, target_fi.d), dtype=np.float32)
target_fi.reconstruct_n(0, target_fi.ntotal, target_all_emb)

for _, row in target_meta_df.iterrows():
    target_emb = torch.tensor(target_all_emb[row['id']], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    generated  = greedy_decode(target_emb)
    truth      = row['text']

    gen_toks    = generated.split()
    truth_toks  = truth.split()
    match_count = sum(g == t for g, t in zip(gen_toks, truth_toks))
    token_acc   = match_count / max(len(gen_toks), len(truth_toks))

    print(f"{row['target_id']}")
    print(f'generated    : {generated}')
    print(f'ground truth : {truth}')
    print(f'token acc    : {token_acc:.0%} | exact : {"yes" if generated == truth else "no"}')
    print()